In [6]:
# Setup: code comes from GitHub onto the VM disk; data + checkpoints live on Drive (see src/config.py)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/s1156052/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN (never write it in the notebook)
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!apt-get -qq install -y cd-hit > /dev/null

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 8 (delta 4), reused 8 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 7.69 KiB | 2.56 MiB/s, done.
From https://github.com/Tienanh209/NeoAntigen
   744fba3..1b549cd  main       -> origin/main
Updating 744fba3..1b549cd
Fast-forward
 docs/lung_study_spec.md  |  28 +++++++
 load_datasets_lung.ipynb |  45 +++++++++--
 src/lung.py              |  65 ++++++++++++----
 src/lung_split.py        | 189 +++++++++++++++++++++++++++++++++++++++++++++++
 4 files changed, 309 insertions(+), 18 deletions(-)
 create mode 100644 docs/lung_study_spec.md
 create mode 100644 src/lung_split.py
/content/s1156052/NeoAntigen


In [7]:
# Download once (dated snapshot on Drive): lung-cancer neoantigens (positives) + viral/TAA epitopes (negatives)
import pandas as pd
from src import config, lung
pd.set_option("display.width", 200)

raw = lung.get_raw(config.DATA_DIR, refresh=False)
print(raw.groupby("group").size())
# Optional extra positives, downloaded by hand into DATA_DIR/lung_extra/ (TSNAdb is *predicted*, not validated):
# extra = lung.load_extra(config.DATA_DIR / "lung_extra" / "tsnadb_lung.tsv", "tsnadb", seq_col="peptide", hla_col="HLA")
# raw = lung.add_extra(raw, [extra])     # then use LungPreset(..., extra_sources=("tsnadb",))
raw = lung.add_extra(raw, [])
lung.sweep(raw)

using snapshot lung_cedar_raw_20261110.csv
group
neo       2472
taa      18161
viral    30488
dtype: int64


,preset,pos,neg,neg/pos
0,lung_cedar,660,48243,73.1
1,lung_cedar_any_assay,2471,48243,19.5


In [8]:
# Run the full pipeline for one preset; output goes to DATA_DIR/<name>/ (data.load_dataset(config.DATA_DIR / PRESET))
PRESET = "lung_cedar"
df, final, tables = lung.run(raw, lung.PRESETS[PRESET], config.DATA_DIR, config.WORK_DIR)
for t in tables:
    print(t.to_string(index=False), "\n")

   Dataset  Epitopes  Similarity <40 %  Length <=35
Neoantigen       660                30           30
    Others     48432              1865         1728
     Total     49092              1895         1758 

   Dataset  Training data (80 %)  Independent data (20 %)
Neoantigen                    24                        6
    Others                  1382                      346
     Total                  1406                      352 

Independent dataset  MHC-I presented peptides  MHC-II presented peptides
         Neoantigen                         1                          5
             Others                         8                        338
              Total                         9                        343 



In [10]:
# Guide steps 3-5: frozen splits (conventional independent test + temporal test), similarity checks, Tables 1-2
# Look at positives per first-evidence year BEFORE fixing T (the guide: choose cutoffs before looking at results)
from src import lung_split
df, info = lung.build(raw, lung.PRESETS[PRESET])
print(lung_split.positives_by_year(df).to_string())

T, E = 2022, 2026          # train/CV: first evidence <= T ; temporal test: (T, E]
res = lung_split.make_splits(df, config.DATA_DIR, config.WORK_DIR, T=T, E=E, params={**info, "preset": PRESET})
for k in ("table1", "table2", "similarity"):
    print(res[k].to_string(), "\n")
print("limitations:", res["limitations"])

KeyError: 'year_first'